# Single Experiment Analysis

Deep-dive into all runs within one experiment. Use this to:
- Compare training curves across seeds/configs
- Identify best and worst runs
- Visualize metric distributions across the sweep
- Check for seed variance and statistical significance

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent.parent.parent
RENDEZVOUS_ROOT = REPO_ROOT / "rendezvous_comm"
sys.path.insert(0, str(RENDEZVOUS_ROOT))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.storage import ExperimentStorage
from src.metrics import compute_m7_sample_efficiency
from src.display import (
    display_config_selector,
    display_sweep_summary,
    display_metric_cards,
    display_verdict,
    scrollable_md,
    display_artifact_tree,
)
from src.plotting import (
    plot_sweep_heatmap,
    plot_seed_variance,
    save_figure,
    set_style,
)

set_style()
print("Ready.")

## 1. Select Experiment

Change `EXP_ID` to analyze a different experiment. The config selector shows all available configs and their completion status.

In [ ]:
EXP_ID = "er1"

display_config_selector(EXP_ID)

storage = ExperimentStorage(EXP_ID)
all_metrics = storage.load_all_metrics()
df = storage.to_dataframe()

print(f"\nCompleted runs: {len(all_metrics)}")
if not df.empty:
    display(
        df[
            [
                "run_id",
                "algorithm",
                "n_agents",
                "lidar_range",
                "seed",
                "M1_success_rate",
                "M2_avg_return",
                "M6_coverage_progress",
            ]
        ].round(3)
    )

## 2. Sweep Heatmaps

Success rate and coverage progress across the sweep grid (N agents × LiDAR range), averaged over seeds.

In [ ]:
if df.empty or len(all_metrics) < 2:
    print("Need at least 2 completed runs for sweep analysis.")
else:
    has_n = "n_agents" in df.columns and df["n_agents"].nunique() > 1
    has_l = "lidar_range" in df.columns and df["lidar_range"].nunique() > 1

    if has_n and has_l:
        for metric in ["M1_success_rate", "M6_coverage_progress", "M2_avg_return"]:
            fig = plot_sweep_heatmap(
                df,
                metric=metric,
                row_param="n_agents",
                col_param="lidar_range",
                title=f"{EXP_ID.upper()}: {metric} (N agents × LiDAR)",
            )
            save_figure(fig, str(storage.results_dir / f"heatmap_{metric}.png"))
            plt.show()
    elif has_n:
        for metric in ["M1_success_rate", "M6_coverage_progress"]:
            fig = plot_seed_variance(
                df,
                metric=metric,
                group_by="n_agents",
                title=f"{EXP_ID.upper()}: {metric} by N agents",
            )
            plt.show()
    elif has_l:
        for metric in ["M1_success_rate", "M6_coverage_progress"]:
            fig = plot_seed_variance(
                df,
                metric=metric,
                group_by="lidar_range",
                title=f"{EXP_ID.upper()}: {metric} by LiDAR range",
            )
            plt.show()
    else:
        print(
            "Only one config in sweep — use the experiment notebook for single-run analysis."
        )

## 3. Algorithm Comparison (MAPPO vs IPPO)

If the sweep includes multiple algorithms, compare them across seeds with error bars.

In [ ]:
if df.empty or "algorithm" not in df.columns or df["algorithm"].nunique() < 2:
    print("Single algorithm — skipping comparison.")
else:
    for metric in [
        "M1_success_rate",
        "M2_avg_return",
        "M3_avg_steps",
        "M6_coverage_progress",
    ]:
        fig = plot_seed_variance(
            df,
            metric=metric,
            group_by="algorithm",
            title=f"{EXP_ID.upper()}: {metric} by Algorithm",
        )
        plt.show()

    # Per-config algo comparison table
    if "n_agents" in df.columns:
        pivot = df.pivot_table(
            index=["n_agents", "lidar_range"],
            columns="algorithm",
            values="M1_success_rate",
            aggfunc=["mean", "std"],
        ).round(3)
        print("Success Rate: mean ± std by config and algorithm")
        display(pivot)

## 4. Seed Variance

How stable are results across random seeds? Low variance = reliable conclusions.

In [ ]:
if df.empty or "seed" not in df.columns or df["seed"].nunique() < 2:
    print("Need multiple seeds for variance analysis.")
else:
    # Group by config (excluding seed) and compute stats
    group_cols = [
        c for c in ["algorithm", "n_agents", "lidar_range"] if c in df.columns
    ]
    if group_cols:
        metrics_to_check = ["M1_success_rate", "M2_avg_return", "M6_coverage_progress"]
        stats = (
            df.groupby(group_cols)[metrics_to_check]
            .agg(["mean", "std", "count"])
            .round(4)
        )
        display(stats)

        # Flag high-variance configs (std > 0.15 on success rate)
        if ("M1_success_rate", "std") in stats.columns:
            high_var = stats[stats[("M1_success_rate", "std")] > 0.15]
            if not high_var.empty:
                print("\nHigh-variance configs (M1 std > 0.15):")
                display(high_var)
            else:
                print("\nAll configs have stable seed variance (M1 std <= 0.15).")

## 5. Training Curves Overlay

Compare training progress across runs. Overlaying curves reveals whether different configs converge at different speeds.

In [ ]:
if not all_metrics:
    print("No completed runs.")
else:
    # Collect training curves from all runs
    run_dirs = storage.list_run_dirs()
    curve_data = {}  # {run_id: {csv_name: [(step, val), ...]}}

    for d in run_dirs:
        from src.storage import _extract_run_id, RunStorage

        rid = _extract_run_id(d.name)
        rs = RunStorage(d, rid)
        scalars = rs.load_benchmarl_scalars()
        if scalars:
            curve_data[rid] = scalars

    if not curve_data:
        print("No BenchMARL scalars found in any run.")
    else:
        # Plot eval reward curves overlaid
        csv_key = "eval_reward_episode_reward_mean"
        fig, ax = plt.subplots(figsize=(12, 5))

        for rid, scalars in curve_data.items():
            data = scalars.get(csv_key)
            if data:
                steps, vals = zip(*data)
                # Short label: extract key params
                label = rid.replace(f"{EXP_ID}_", "")
                ax.plot(steps, vals, linewidth=1.2, alpha=0.7, label=label)

        ax.set_xlabel("Iteration")
        ax.set_ylabel("Eval Reward")
        ax.set_title(f"{EXP_ID.upper()}: Training Curves (all runs)")
        ax.legend(fontsize=8, ncol=2, loc="lower right")
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        save_figure(fig, str(storage.results_dir / "training_curves_overlay.png"))
        plt.show()

## 6. Sample Efficiency (M7)

Frames to reach 80% of final eval reward. Lower = faster learning.

In [ ]:
if not all_metrics:
    print("No completed runs.")
else:
    m7_data = []
    for d in storage.list_run_dirs():
        from src.storage import _extract_run_id

        rid = _extract_run_id(d.name)
        m7 = compute_m7_sample_efficiency(d)
        if m7 is not None:
            m7_data.append({"run_id": rid, "M7_frames_to_80pct": m7})

    if m7_data:
        m7_df = pd.DataFrame(m7_data)
        display(m7_df.sort_values("M7_frames_to_80pct"))

        fig, ax = plt.subplots(figsize=(10, max(3, len(m7_df) * 0.4)))
        labels = [r.replace(f"{EXP_ID}_", "") for r in m7_df["run_id"]]
        ax.barh(labels, m7_df["M7_frames_to_80pct"] / 1e6, color="#3498db", height=0.6)
        ax.set_xlabel("Frames to 80% of Final Reward (millions)")
        ax.set_title(f"{EXP_ID.upper()}: Sample Efficiency (M7)")
        ax.grid(True, alpha=0.2, axis="x")
        fig.tight_layout()
        plt.show()
    else:
        print("Could not compute M7 for any run (need eval reward + frames CSVs).")

## 7. Best and Worst Runs

Identify the configurations that performed best and worst on success rate.

In [ ]:
if df.empty:
    print("No completed runs.")
else:
    ranked = df.sort_values("M1_success_rate", ascending=False)
    show_cols = [
        "run_id",
        "M1_success_rate",
        "M2_avg_return",
        "M3_avg_steps",
        "M6_coverage_progress",
    ]
    show_cols = [c for c in show_cols if c in ranked.columns]

    n_show = min(5, len(ranked))
    print(f"Top {n_show} runs:")
    display(ranked.head(n_show)[show_cols].round(3))

    if len(ranked) > n_show:
        print(f"\nBottom {n_show} runs:")
        display(ranked.tail(n_show)[show_cols].round(3))

    # KPI cards for best run
    best_id = ranked.iloc[0]["run_id"]
    best_metrics = all_metrics[best_id]
    display_metric_cards(best_metrics, title=f"Best: {best_id}")

## 8. Metric Distributions

Box plots showing the spread of each metric across all runs.

In [ ]:
if df.empty or len(df) < 3:
    print("Need at least 3 runs for distribution plots.")
else:
    metric_cols = [c for c in df.columns if c.startswith("M") and c[1].isdigit()]
    n_metrics = len(metric_cols)
    ncols = min(3, n_metrics)
    nrows = (n_metrics + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = np.array(axes).flatten() if n_metrics > 1 else [axes]

    for ax, col in zip(axes, metric_cols):
        data = df[col].dropna()
        ax.boxplot(
            data,
            vert=True,
            patch_artist=True,
            boxprops=dict(facecolor="#3498db", alpha=0.6),
        )
        ax.set_title(col, fontsize=11)
        ax.grid(True, alpha=0.2)

        # Add individual points
        jitter = np.random.normal(1, 0.04, size=len(data))
        ax.scatter(jitter, data, alpha=0.5, s=20, color="#e74c3c", zorder=5)

    # Hide unused axes
    for ax in axes[n_metrics:]:
        ax.set_visible(False)

    fig.suptitle(
        f"{EXP_ID.upper()}: Metric Distributions (n={len(df)} runs)",
        fontsize=14,
        fontweight="bold",
    )
    fig.tight_layout()
    save_figure(fig, str(storage.results_dir / "metric_distributions.png"))
    plt.show()

## 9. Full Sweep Summary Table

All completed runs with their metrics, sorted by success rate.

In [ ]:
if all_metrics:
    display_sweep_summary(all_metrics)

    # Verdict based on aggregate results
    if not df.empty:
        avg_success = df["M1_success_rate"].mean()
        avg_return = df["M2_avg_return"].mean()
        display_verdict(avg_success, avg_return)

## 10. Run Reports

Browse individual run reports (markdown). Change the run index to view different runs.

In [ ]:
RUN_INDEX = 0  # Change to browse different runs

if all_metrics:
    run_ids = sorted(all_metrics.keys())
    if RUN_INDEX < len(run_ids):
        rid = run_ids[RUN_INDEX]
        rs = storage.get_run(rid)

        report_path = rs.run_dir / "report.md"
        if report_path.exists():
            scrollable_md(report_path.read_text(), title=f"Report: {rid}")
        else:
            print(f"No report.md found for {rid}")

        display_artifact_tree(rs)
    else:
        print(f"Run index {RUN_INDEX} out of range (0-{len(run_ids)-1})")
else:
    print("No completed runs.")